# Flowers-102 — fine-tuning ResNet-50 and exporting to ONNX

A notebook for **Google Colab with a GPU**. It produces two files that belong
in the project's `model/` folder:

- `flowers102.onnx` — the network with normalisation and softmax inside the graph
- `flowers102.labels.json` — class names, metrics and the preprocessing spec

### Before you start

`Runtime` → `Change runtime type` → **T4 GPU** (or any GPU on offer).

Then `Runtime` → `Run all`. A full pass on a T4 takes about 10 minutes, roughly
6 of them training; the dataset (~345 MB) downloads in half a minute.

Our run reached 86.5% top-1 after the first phase and 95.3% after the second.

### What happens here

1. Take `resnet50` with ImageNet weights.
2. **Phase 1:** freeze the backbone, train only the new 102-class `fc`.
3. **Phase 2:** unfreeze `layer3` + `layer4` and fine-tune with a small lr.
4. Measure top-1 on the official `test` split (6149 images).
5. Export to ONNX and compare torch against onnxruntime.

Training runs on `train` + `val` (2040 images, 20 per class) — the usual
practice for this dataset, where the test split is three times the size of
the training one.

## 0. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU FOUND: Runtime -> Change runtime type -> T4 GPU"

Colab ships without `onnx`, which the exporter needs, and without
`onnxruntime`, which the parity check needs. Install both up front: doing it
mid-run may make pip ask for a session restart, and the trained model would
go with it.

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
import json, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import transforms
from torchvision.datasets import Flowers102
from torchvision.models import resnet50, ResNet50_Weights

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: training on a CPU takes hours. Enable the GPU in the runtime settings.")

## 1. Dataset

`download=True` pulls the archive from Oxford's servers. Colab caches it only
until the session restarts — a later run downloads it again.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224
RESIZE_SIZE = 256

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

ROOT = "./data"
train_part = Flowers102(ROOT, split="train", transform=train_tf, download=True)
val_part = Flowers102(ROOT, split="val", transform=train_tf, download=True)
test_set = Flowers102(ROOT, split="test", transform=eval_tf, download=True)

train_set = ConcatDataset([train_part, val_part])
print(f"train+val: {len(train_set)} | test: {len(test_set)}")

In [ ]:
# Class names. Recent torchvision versions expose them on Flowers102;
# older ones do not, hence the fallback list.
CLASS_NAMES_FALLBACK = [
    "pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
    "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
    "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
    "yellow iris", "globe-flower", "purple coneflower", "peruvian lily",
    "balloon flower", "giant white arum lily", "fire lily", "pincushion flower",
    "fritillary", "red ginger", "grape hyacinth", "corn poppy", "prince of wales feathers",
    "stemless gentian", "artichoke", "sweet william", "carnation", "garden phlox",
    "love in the mist", "mexican aster", "alpine sea holly", "ruby-lipped cattleya",
    "cape flower", "great masterwort", "siam tulip", "lenten rose", "barbeton daisy",
    "daffodil", "sword lily", "poinsettia", "bolero deep blue", "wallflower", "marigold",
    "buttercup", "oxeye daisy", "common dandelion", "petunia", "wild pansy", "primula",
    "sunflower", "pelargonium", "bishop of llandaff", "gaura", "geranium",
    "orange dahlia", "pink-yellow dahlia", "cautleya spicata", "japanese anemone",
    "black-eyed susan", "silverbush", "californian poppy", "osteospermum",
    "spring crocus", "bearded iris", "windflower", "tree poppy", "gazania", "azalea",
    "water lily", "rose", "thorn apple", "morning glory", "passion flower", "lotus",
    "toad lily", "anthurium", "frangipani", "clematis", "hibiscus", "columbine",
    "desert-rose", "tree mallow", "magnolia", "cyclamen", "watercress", "canna lily",
    "hippeastrum", "bee balm", "ball moss", "foxglove", "bougainvillea", "camellia",
    "mallow", "mexican petunia", "bromelia", "blanket flower", "trumpet creeper",
    "blackberry lily",
]

CLASS_NAMES = list(getattr(test_set, "classes", None) or CLASS_NAMES_FALLBACK)
NUM_CLASSES = len(CLASS_NAMES)
assert NUM_CLASSES == 102, f"expected 102 classes, got {NUM_CLASSES}"
print(f"{NUM_CLASSES} classes, first five: {CLASS_NAMES[:5]}")

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

Look at what actually reaches the network after augmentation — a cheap
safeguard against images that end up rotated or cropped beyond recognition.

In [ ]:
import matplotlib.pyplot as plt

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for ax in axes.flat:
    img, label = train_set[random.randrange(len(train_set))]
    ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[label], fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Model

In [ ]:
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

def set_backbone_trainable(model, names):
    """Unfreeze only the listed top-level blocks (fc always trains)."""
    for param in model.parameters():
        param.requires_grad = False
    for name in list(names) + ["fc"]:
        for param in getattr(model, name).parameters():
            param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable parameters: {trainable:,} of {total:,} ({100 * trainable / total:.1f}%)")

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

## 3. Training and evaluation loop

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.numel()
    return correct / total


def train_phase(model, epochs, lr, trainable_blocks, label):
    set_backbone_trainable(model, trainable_blocks)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader))
    scaler = torch.amp.GradScaler(DEVICE, enabled=(DEVICE == "cuda"))

    for epoch in range(1, epochs + 1):
        model.train()
        started = time.time()
        running_loss = seen = 0
        for images, labels in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=(DEVICE == "cuda")):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item() * labels.numel()
            seen += labels.numel()
        print(f"[{label}] epoch {epoch}/{epochs}  loss {running_loss / seen:.4f}"
              f"  {time.time() - started:.0f} s")
    return model

### Phase 1 — the classifier only

The backbone is frozen and the new `fc` trains from scratch. This moves the
head away from its random initialisation quickly, so that in the next phase
large gradients do not wreck the pretrained features.

In [ ]:
train_phase(model, epochs=8, lr=1e-3, trainable_blocks=[], label="phase 1")
acc_head_only = evaluate(model, test_loader)
print(f"\ntop-1 after phase 1: {acc_head_only:.4f}")

### Phase 2 — unfreezing the upper blocks

`layer3` and `layer4` hold the most specific features, and those are what we
adapt to flowers. The early layers (edges, textures) stay as they are —
fine-tuning them on 2040 images would only hurt.

In [ ]:
train_phase(model, epochs=15, lr=1e-4, trainable_blocks=["layer3", "layer4"], label="phase 2")
acc_final = evaluate(model, test_loader)
print(f"\ntop-1 after phase 1: {acc_head_only:.4f}")
print(f"top-1 after phase 2: {acc_final:.4f}")

Save the weights before exporting: if anything breaks further down, the
training does not have to be repeated — the state can be loaded back.

In [ ]:
torch.save(model.state_dict(), "resnet50_flowers102.pt")
print("weights saved: resnet50_flowers102.pt")

# To restore after a session restart:
# model.load_state_dict(torch.load("resnet50_flowers102.pt"))

## 4. Exporting to ONNX

The key decision: **ImageNet normalisation and softmax live inside the graph**.
The website runs without torch and torchvision, so any drift in preprocessing
would only surface in production. That makes the contract with the server as
small as it gets: hand it an RGB tensor `1x3x224x224` with values in `[0, 1]`
and the model does the rest.

In [ ]:
class ExportWrapper(nn.Module):
    """Takes RGB in [0,1], normalises inside and returns probabilities."""

    def __init__(self, model, mean, std):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, pixel_values):
        normalized = (pixel_values - self.mean) / self.std
        return torch.softmax(self.model(normalized), dim=1)


wrapper = ExportWrapper(model, IMAGENET_MEAN, IMAGENET_STD).to("cpu").eval()
dummy = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)

ONNX_PATH = "flowers102.onnx"
export_kwargs = dict(
    input_names=["pixel_values"],
    output_names=["probabilities"],
    opset_version=17,
    dynamic_axes={"pixel_values": {0: "batch"}, "probabilities": {0: "batch"}},
)
try:
    torch.onnx.export(wrapper, dummy, ONNX_PATH, dynamo=False, **export_kwargs)
except TypeError:
    torch.onnx.export(wrapper, dummy, ONNX_PATH, **export_kwargs)

print(f"{ONNX_PATH}: {Path(ONNX_PATH).stat().st_size / 1e6:.1f} MB")

### Parity check

Run the same images through torch and through onnxruntime. If the
probabilities agree, the export is sound and torch can be forgotten.

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

# Denormalise a test batch back to [0,1] — exactly what the server sends.
images, labels = next(iter(test_loader))
batch = (images[:8] * std + mean).clamp(0, 1)

with torch.no_grad():
    torch_probs = wrapper(batch).numpy()
onnx_probs = session.run(None, {"pixel_values": batch.numpy()})[0]

max_diff = np.abs(torch_probs - onnx_probs).max()
print(f"largest probability difference: {max_diff:.2e}")
assert max_diff < 1e-4, "torch and onnxruntime disagree — the export is broken"

print("\nonnxruntime predictions:")
for i in range(len(batch)):
    top = onnx_probs[i].argmax()
    mark = "+" if top == labels[i].item() else "-"
    print(f"  {mark} {CLASS_NAMES[top]:<28} {onnx_probs[i][top]:.3f}"
          f"   (truth: {CLASS_NAMES[labels[i].item()]})")

## 5. Metadata and download

The description next to the model carries everything the server needs,
preprocessing included — so changing the architecture does not mean editing
the website's code.

In [ ]:
meta = {
    "architecture": "resnet50",
    "dataset": "Flowers102",
    "num_classes": NUM_CLASSES,
    "classes": CLASS_NAMES,
    "accuracy": {
        "test_top1_head_only": round(acc_head_only, 4),
        "test_top1_finetuned": round(acc_final, 4),
        "test_size": len(test_set),
    },
    "preprocessing": {
        "resize_shorter_side": RESIZE_SIZE,
        "center_crop": IMG_SIZE,
        "scale": "divide by 255, range [0,1]",
        "normalization": "inside the ONNX graph",
        "input_name": "pixel_values",
        "input_shape": [1, 3, IMG_SIZE, IMG_SIZE],
        "output_name": "probabilities",
        "output": "softmax probabilities",
    },
}

Path("flowers102.labels.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print(json.dumps({k: v for k, v in meta.items() if k != "classes"}, indent=2, ensure_ascii=False))

In [ ]:
from google.colab import files

files.download("flowers102.onnx")
files.download("flowers102.labels.json")

Both files belong in the project's `model/` folder:

```
model/
  flowers102.onnx
  flowers102.labels.json
```

If Colab blocks the download (Safari and strict cookie settings do that),
save to Drive instead:

```python
from google.colab import drive
drive.mount("/content/drive")
!cp flowers102.onnx flowers102.labels.json /content/drive/MyDrive/
```